In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import re
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

In [2]:
homePath = Path(r"D:\ThesisData\back_notrees")

In [3]:
trees = False

In [ ]:
for folder in homePath.glob("*"):
    for subfolder in folder.glob("*"):
        data = pd.DataFrame(columns=["grviIn", "meanRefR", "meanRefG", "illumR", "illumG", "blockIdx", "sunRoll", "sunPitch", "sunYaw", "camRoll", "camPitch", "camYaw"])
        
        if trees:
            print("not yet")
            break
        else:
            with open(str(subfolder) + r"\EnvParams.txt", "r") as f:
                illuminanceFile = f.readlines()
            
            red_raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', illuminanceFile[30])
            red = float(red_raw[0]) + float("0." + str(red_raw[1]))

            green_raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', illuminanceFile[31])
            green = float(green_raw[0]) + float("0." + str(green_raw[1]))

            blue_raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', illuminanceFile[32])
            blue = float(blue_raw[0]) + float("0." + str(blue_raw[1]))

            #debug
            #print (red, green, blue)

        sunPos_raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', illuminanceFile[24])

        sunPos_roll = float(sunPos_raw[0])
        sunPos_pitch = float(str(sunPos_raw[1]) + "." + str(sunPos_raw[2]))
        sunPos_yaw = float(str(sunPos_raw[3]) + "." + str(sunPos_raw[4]))
        
        with open (str(subfolder) + r"\ObjectPoses.txt", "r") as f:
            camRotationFile = f.readlines()

        # this saves STRINGS
        rotations = []
        for i in range (0, len(camRotationFile)):
            step = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', camRotationFile[i])
            step = step[len(step)-3 : len(step)]

            rotations.append(step)

        processedFolder = Path(str(subfolder) + r"\Processed")

        files = sorted (
            processedFolder.glob("proc_*.npy"),
            key=lambda p: int(p.stem.split("_")[1])
        )
        
        for i in range (0, len(files)):
            blocks = np.load(str(files[i]))
            currRot = rotations[i]

            for j in range (0, len(blocks)):
                img = blocks[j]
                
                r_ref = (img[:, :, 0] * np.pi)/red 
                g_ref = (img[:, :, 1] * np.pi)/green

                grvi = (g_ref - r_ref)/(g_ref + r_ref + 0.00000001)

                grvi_m = grvi.mean()

                data.loc[len(data)] = [grvi_m, r_ref.mean(), g_ref.mean(), red, green, j, sunPos_roll, sunPos_pitch, sunPos_yaw, float(currRot[0]), float(currRot[1]), float(currRot[2])]

        data.to_csv(str(subfolder) + r"\data.csv", index=False)

In [ ]:
subfolder